# Davinci — Near-duplicate candidate review

A separate review notebook that reports near-duplicate candidate pairs for human inspection. It does not automatically delete or merge records.

In [ ]:
import json, subprocess, sys
from pathlib import Path
DATA=Path('/content/drive/MyDrive/Davinci/data_pipeline/processed/persian_corpus_merged.jsonl')
OUT=Path('/content/drive/MyDrive/Davinci/data_pipeline/audits/near_duplicate_candidates.jsonl'); OUT.parent.mkdir(parents=True,exist_ok=True)
REPO=Path('/content/Davinci')
if not REPO.exists(): subprocess.run(['git','clone','--depth','1','https://github.com/keyhmoham/Davinci.git',str(REPO)],check=True)
else: subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],check=True)
sys.path.insert(0,str(REPO)); from src.davinci.near_dedup import find_near_duplicate_pairs
assert DATA.is_file(),f'Missing corpus: {DATA}'


## Parameters

This implementation uses word n-gram SimHash/LSH candidate discovery followed by Jaccard verification. It is approximate; LSH may miss pairs, and candidates require manual review. Processing stores shingle sets in memory, so begin with a bounded sample.

In [ ]:
MAX_RECORDS=50000 # deliberately bounded initial run; increase only after memory assessment
JACCARD_THRESHOLD=0.85; HAMMING_DISTANCE=6; NGRAM_SIZE=5
records=[]
with DATA.open(encoding='utf-8') as f:
 for line in f:
  if not line.strip(): continue
  obj=json.loads(line); text=obj.get('text')
  if isinstance(text,str) and text.strip(): records.append({'text':text,'source_row':obj.get('source_row',len(records))})
  if len(records)>=MAX_RECORDS: break
print('Records loaded:',len(records))


In [ ]:
result=find_near_duplicate_pairs(records,jaccard_threshold=JACCARD_THRESHOLD,max_hamming_distance=HAMMING_DISTANCE,ngram_size=NGRAM_SIZE)
print({k:v for k,v in result.items() if k!='pairs'})
with OUT.open('w',encoding='utf-8') as f:
 for pair in result['pairs']: f.write(json.dumps(pair,ensure_ascii=False)+'\n')
print('Candidates:',len(result['pairs']),'export:',OUT)


## Review policy

Do not auto-delete candidates. Compare content, source/license metadata, and provenance. For large corpora, adjust sampling and thresholds, document the run parameters, and consider a distributed or disk-backed index before increasing memory-intensive processing.